# AI Engineering Interview Prep
## Section: AI System Design

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 461+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "⚡ Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 🏗️ Section 7 — AI System Design

### Design 1: Design ChatGPT (Training to Serving, End to End)
**A:**
**Training Pipeline:**
- Pre-training: cluster of thousands of GPUs, distributed training (FSDP/Megatron) on trillion-token corpus, cross-entropy next-token loss
- SFT: instruction-tuning on 100K+ high-quality human-written demos
- Reward Model: train on 1M+ human preference pairs
- RLHF/PPO: optimise against reward model with KL constraint

**Serving Infrastructure:**
- Model sharding across GPU nodes (tensor + pipeline parallelism)
- vLLM / TensorRT-LLM for optimised inference (continuous batching, paged attention)
- Load balancer → inference cluster (auto-scaled) → response streaming
- KV cache management per request
- Content moderation layer (input + output)
- Rate limiting, authentication, billing

**Scale:** OpenAI serves 100M+ users → millions of concurrent requests → global multi-region deployment, CDN for API responses, Redis for rate limiting

🏷️ *My production discipline from Yotta (monitoring, autoscaling, failover) applies here — the LLM inference layer is just another stateless service.*


### Design 2: Design a RAG System (Chat with Your Documents)
**A:**
**Offline Ingestion Pipeline:**
```
Documents → Document Loader → Chunker (512 tokens, 50 overlap)
→ Embedding Model → Vector DB (FAISS/Pinecone) + Metadata Store
```

**Online Query Pipeline:**
```
User Query → Auth → Query Embedding → Hybrid Search (Vector + BM25)
→ Cross-encoder Reranker (top-3) → Context Builder
→ LLM (Gemini/GPT-4) with system prompt → Structured Response
→ Citation Attribution → User
```

**Key design decisions:**
- Chunk size: 512 tokens (production tested sweet spot)
- Hybrid retrieval: BM25 for exact matches + dense for semantic
- Reranker: cross-encoder for quality-critical applications
- Prompt caching: cache system prompt + frequently retrieved contexts
- Streaming: stream tokens for low latency UX

🏷️ *This is Nyaya-Pro's exact architecture. FastAPI backend, FAISS, LangChain, Gemini, sentence-transformers, cross-encoder — all production.*


### Design 3: Design Memory for a Personal AI Assistant
**A:**
**Memory tiers:**

**Short-term (in-context):** Last 5-10 conversation turns. Stored in session state (Redis). Automatically included in every LLM call. Lost when session expires.

**Mid-term (episodic buffer):** Compressed summaries of recent conversations. "Last week the user asked about investing strategies." Stored in DB, retrieved and prepended to context.

**Long-term semantic memory:** User facts, preferences, relationships. "User is a lawyer. Prefers concise answers. Has 2 kids." Stored as embeddings in vector DB + structured fields in SQL. Retrieved based on relevance to current query.

**Procedural memory:** System prompt — defines the assistant's behaviour and persona. Updated manually.

**Retrieval strategy:** On each query, retrieve: (1) relevant facts from semantic memory, (2) relevant past conversations, (3) recent short-term context. Build a context that's < 4K tokens.

**Privacy:** Separate memory per user. Encryption at rest. User can delete their memory.


### Design 4: Design a Deep Research Agent
**A:**
**Architecture:**

```
User Query
→ Research Planner Agent (GPT-4): generate research plan with sub-questions
→ Parallel SubAgents:
   ├── Web Search Agent (SerpAPI/Tavily) 
   ├── Academic Search Agent (Semantic Scholar API)
   └── Knowledge Base Agent (internal vector DB)
→ Synthesis Agent: merge findings, identify gaps
→ Gap Analysis: are there unanswered sub-questions?
   → If yes: spawn additional targeted search agents
→ Report Generator: structured markdown report with citations
→ Fact Checker Agent: verify key claims
→ Final Report with inline citations
```

**Key components:**
- Max_steps guardrail (prevents infinite loops)
- Source credibility scoring
- Citation extraction and verification
- Progressive disclosure (partial results streamed)
- Cost tracking per research session

🏷️ *Similar to what I'd build by extending Nyaya-Pro — for legal research, add Indian court case databases and gazette notifications as search sources.*


### Design 5: Design a Multi-Agent Customer Support System
**A:**
```
User Message
→ Intent Classifier (fast, cheap model): classify type + urgency
→ Tier Router:
   ├── FAQ Agent (70% volume): embedding search over FAQ KB → answer
   ├── Account Agent (CRM lookup): retrieve account info, billing, orders → resolve
   ├── Technical Agent (KB + product docs): deep RAG on technical KB → resolve
   └── Human Escalation (10%): create ticket, notify human, handoff context
→ Response Quality Check (guardrails)
→ CSAT Collection
→ Feedback → Fine-tuning pipeline
```

**Escalation triggers:** negative sentiment, 2 failed resolution attempts, keywords (legal/safety), user request.

**State management:** each conversation has a persistent state (Redis): tier, previous_agents_tried, resolution_attempts, user_sentiment_score.

**Metrics:** containment rate, resolution rate, CSAT score, escalation rate, average handle time.


### Design 6: Design an On-Device AI Assistant
**A:**
**Constraints:** CPU/mobile NPU only (4-8GB RAM), battery-sensitive, privacy-critical, no internet dependency.

**Model selection:** SLM optimised for edge — Phi-3 Mini (3.8B), Llama 3 8B 4-bit, Apple's OpenELM, Google Gemma 2B.

**Optimisation stack:**
- 4-bit quantisation (GGUF format for CPU, CoreML for Apple silicon)
- llama.cpp / MLC-LLM for CPU inference
- Speculative decoding (small draft model + larger verifier)
- Prompt caching for repeated prefix (system prompt)

**Architecture:**
- Local SQLite for conversation history
- On-device vector search (FAISS with small index for personal notes)
- Hybrid: on-device model for privacy-sensitive tasks; cloud API for complex reasoning (user opt-in)
- Wake word detection (on-device, always-on tiny model)
- STT/TTS on-device (Whisper Tiny, Kokoro-TTS)


### Design 7: Design a Multimodal Search System (Text, Image, Video)
**A:**
**Indexing Pipeline:**
- Text: chunk → text embeddings → vector DB
- Images: ViT/CLIP → image embeddings → same vector DB (shared space)
- Video: extract keyframes every N seconds → image embeddings; transcribe audio → text embeddings
- Metadata: stored alongside all embeddings

**Query Pipeline:**
- Text query → CLIP text encoder → vector search (finds both matching text AND image/video content)
- Image query → CLIP image encoder → vector search
- Voice query → Whisper STT → text query pipeline

**Re-ranking:** Multi-modal reranker that scores text+image relevance together

**Architecture:**
- Ingestion: async Kafka pipeline (video processing takes minutes)
- Index: Weaviate (supports multi-modal natively) or Pinecone with content-type metadata
- API: GraphQL for flexible field selection
- CDN: serve media, not from DB

**Scale:** YouTube-scale → sharded index, distributed video processing workers, GPU cluster for embedding


### Design 8: Design an LLM Inference Platform (vLLM-as-a-Service)
**A:**
```
Client → API Gateway (rate limiting, auth, billing) 
→ Request Router (model selection, priority queue)
→ Inference Cluster:
   ├── GPU Node 1: vLLM engine (continuous batching, paged attention)
   ├── GPU Node 2: vLLM engine
   └── GPU Node N: vLLM engine
→ Auto-scaler (Kubernetes HPA on GPU utilisation)
→ Monitoring (Prometheus, Grafana): TTFT, TPS, GPU util
→ Cost Metering → Billing
```

**Key engineering decisions:**
- **Continuous batching**: multiple requests share GPU at token level (vLLM default)
- **Paged attention**: KV cache as pages — no memory waste, supports variable-length requests
- **Model sharding**: tensor parallelism for large models (70B+ across multiple GPUs)
- **Preemption**: lower-priority requests yield KV cache pages to high-priority requests
- **Speculative decoding**: draft model generates candidates, target model verifies — faster output

**Metrics to track:** TTFT, ITL (inter-token latency), throughput (tok/sec), GPU MFU (model FLOP utilisation)


### Design 9: Design an LLM Evaluation Platform
**A:**
```
Evaluation Request → Eval Orchestrator
→ Test Suite Runner:
   ├── Automated metrics (BLEU, ROUGE, BERTScore, Perplexity)
   ├── LLM-as-judge (GPT-4 rates outputs 1-5 on multiple dimensions)
   ├── Golden dataset comparison (exact match against reference answers)
   └── Task-specific evaluation (code execution, RAG metrics, agent trajectory)
→ Result Aggregator → Dashboard (Weights & Biases / LangSmith)
→ Regression detector (alert if score drops vs baseline)
→ Report generation → Slack/email notification
```

**Golden datasets:** domain-specific test cases with human-verified answers. Version-controlled.

**A/B comparison:** run two models/prompts on the same test set; compute statistical significance of score difference.

**Human eval pipeline:** disagreement cases sent to human annotators via Label Studio.

**Continuous evaluation:** trigger on every code/prompt change in CI/CD pipeline.


### Design 10: Design a Text-to-Image Generation Service (Midjourney-like)
**A:**
```
User Prompt
→ Prompt Enhancement Agent (LLM): expand and enrich the prompt for better images
→ Safety Classifier: block NSFW, harmful, IP-infringing requests
→ Image Generation Queue (async):
   ├── Stable Diffusion / FLUX model on GPU cluster
   ├── CFG-guided diffusion (20-50 denoising steps)
   └── ControlNet / LoRA adapters for style control
→ Post-processing: upscaling, watermarking, metadata injection
→ CDN storage (S3 + CloudFront)
→ Webhook / SSE notification to client
```

**Scale considerations:**
- Image generation: 5-30 seconds per image on A100. Use a GPU queue with priority scheduling.
- Cache: if the exact same prompt was generated before, return cached result
- Batch generation: 4 variants per prompt (diversity without extra latency)
- Cost: ~$0.01-0.05 per image on GPU spot instances

**Safety:** pre-generation classifier (block requests) + post-generation detector (block results) + abuse monitoring


### Design 11: Design a Music Generation Service (Suno-like)
**A:**
- **Input processing**: text prompt + optional audio reference → encoded embeddings
- **Music generation model**: transformer-based audio token generation (EnCodec + language model, like AudioCraft/MusicGen)
- **Generation pipeline**: text → music tokens → audio codec decode → WAV → MP3
- **Duration control**: variable-length generation with looping/extension capability
- **Post-processing**: mastering pipeline (normalisation, EQ), format conversion
- **CDN delivery**: generated audio served from S3 + CloudFront with streaming support

**Latency:** 10-60 seconds for 30-60 second music clips. Present progress bar + stream partial audio.

**Storage:** ~5MB per 1-minute audio file (MP3). At 1M generations/day = 5TB/day — use S3 Glacier for cold storage after 30 days.

**Licensing:** embed watermark in generated audio for IP tracking.


### Design 12: Design a Video Generation Service (Sora-like)
**A:**
- **Architecture**: Video diffusion model (DiT — Diffusion Transformer) or autoregressive video token model
- **Generation pipeline**: text + optional image → video latent → video decoder → MP4
- **Compute**: extremely expensive — 10-100 GPU-minutes per 60-second video
- **Queue system**: priority queue with ETA communication to users
- **Progressive preview**: generate keyframes first → deliver low-res preview → full resolution in background
- **Storage**: 100MB-1GB per video → S3 + aggressive CDN caching
- **Safety**: pre-generation content classifier + NSFW/deepfake post-generation detector
- **Scaling**: auto-scale GPU cluster (Kubernetes + GPU spot instances); autoscale based on queue depth
- **Watermarking**: C2PA content authenticity standard to mark AI-generated video


### Design 13: Design an AI Coding Agent
**A:**
```
User Request (natural language or buggy code)
→ Intent Classifier: code generation / bug fix / refactor / explain
→ Context Gatherer:
   ├── Repository search (code RAG on codebase embeddings)
   ├── Documentation retrieval
   └── Error message parsing
→ Code Generation Agent (Claude/GPT-4):
   ├── Plan: outline the approach
   ├── Generate: write code with tests
   └── Self-review: critique and improve
→ Sandboxed Execution Environment (Docker):
   ├── Run tests
   ├── Linting/type checking
   └── Return stdout/stderr + test results
→ Reflection loop: if tests fail, analyse and fix (max 5 iterations)
→ PR/diff generation
→ Human review (HITL)
```

🏷️ *This is essentially what I would build as a senior feature on top of Nyaya-Pro's agentic architecture — substituting legal tools for code tools.*


### Design 14: Design a code generation and review system.
**A:**
**Code Generation:**
- User describes feature → LLM generates code with inline comments and tests
- Language detection → route to language-specific fine-tuned model
- RAG over codebase: retrieve similar existing code to maintain style consistency
- Generate: implementation + unit tests + docstring

**Code Review:**
- Diff ingestion → chunk by file/function
- Parallel review agents: security vulnerabilities, performance issues, style/best practices, bug detection
- Each agent comments on relevant lines
- Aggregator synthesises across agents
- Severity classification: blocker vs suggestion
- Auto-fix suggestions for common issues

**Integration:**
- GitHub App: triggered on PR creation
- Post review comments directly to PR via GitHub API
- Learning loop: track which comments are accepted/rejected → improve future reviews


### Design 15: Design a content moderation system using AI.
**A:**
```
Content Input (text/image/video)
→ Fast pre-filter (rule-based): block known bad content by hash/pattern (ms latency)
→ ML Classifier: multi-label classification (hate, NSFW, spam, violence, misinformation)
   ├── Text: fine-tuned BERT/RoBERTa
   ├── Image: ViT-based classifier
   └── Video: keyframe sampling + image classifier
→ LLM context analysis (for borderline cases): "Is this satirical? Is this newsworthy?"
→ Decision engine:
   ├── High confidence block → auto-remove
   ├── Medium confidence → human review queue
   └── Low confidence → allow + monitor
→ Human review queue (Label Studio): domain experts + appeals
→ Feedback loop → retrain classifier
```

**Scale:** major platforms process millions of posts/hour → edge servers for pre-filtering, GPU cluster for ML classifiers, human review for escalations.


### Design 16: Design a real-time AI recommendation system.
**A:**
```
User Event Stream (clicks, views, purchases) → Kafka → Feature Store (Feast)
→ Online Serving:
   ├── Candidate retrieval: ANN search on item embeddings (Pinecone/FAISS)
   ├── LLM re-ranking: "Given this user's history, rank these 100 items"
   └── Business rules: boost sponsored, filter out-of-stock
→ Response: ranked item list (< 100ms)

Offline Training:
→ User events → Feature engineering → Two-tower model training
   (user encoder × item encoder, trained on click/purchase signals)
→ Item embeddings updated nightly → vector DB refresh
→ LLM fine-tuned on user preference pairs (DPO)
```

**LLM's role in recommendations:** Understanding complex natural language product descriptions, handling cold-start for new items (describe → embed), and personalising for nuanced user preferences (contextual).


### Design 17: Design an AI-powered email assistant.
**A:**
**Features:** inbox triage (priority classification), smart reply drafting, meeting scheduling, follow-up reminders, unsubscribe recommendations.

**Architecture:**
```
Email Ingestion (IMAP/Gmail API) → Feature extraction
→ Classification agent: urgent / FYI / action-required / spam
→ Summary agent: 3-bullet summary for each email thread
→ Reply drafter: user clicks "draft reply" → 
   context = (thread history + user's past emails for style) →
   LLM drafts response matching user's voice
→ Calendar agent: detects meeting requests → checks calendar → proposes times
→ Smart notifications: only interrupt for urgent + relevant emails
```

**Privacy:** All processing on-device or in user's private cloud tenant. No email content sent to shared LLM infrastructure. Use local SLM for privacy-sensitive drafting.


### Design 18: Design a medical diagnosis assistant using AI.
**A:**
**Critical constraint:** This is a decision-support tool, NOT a replacement for doctors. Every output must emphasise this.

```
Patient symptoms + medical history (EHR integration) + test results
→ Safety classifier: emergency detection (rule-based: chest pain + radiation → flag immediately)
→ Differential diagnosis agent (medical LLM fine-tuned on clinical text)
→ Evidence retrieval (RAG over PubMed, clinical guidelines, drug interactions)
→ Structured output: {possible_diagnoses: [...], evidence: [...], recommended_tests: [...], urgency: "high/medium/low"}
→ Confidence calibration: flag when confidence is low
→ Mandatory disclaimer: "Consult a licensed physician before acting on this information"
→ Doctor review interface (HITL)
→ Audit trail (every suggestion logged with timestamp and model version)
```

**Regulatory:** FDA Software as Medical Device (SaMD) compliance. HIPAA for PII. EU MDR.


### Design 19: Design a fraud detection system powered by LLMs.
**A:**
**Real-time transaction scoring:**
```
Transaction event → Feature store → Fast ML model (XGBoost) → risk score
→ If risk > threshold → LLM analysis:
   "Explain why this transaction is suspicious: [transaction features]"
   → LLM assesses: matches known fraud patterns? User's behaviour deviation?
→ Decision: approve / decline / step-up authentication
→ Explanation generation for declined transactions (regulatory requirement)
```

**LLM's role:** Pattern explanation, novel fraud detection (LLM can identify patterns not in training data by reasoning about them), generating human-readable explanations for compliance.

**Traditional ML vs LLM:** XGBoost for speed (<10ms scoring), LLM for explanation and novel pattern analysis (100-500ms, only for flagged transactions).

**Feedback loop:** fraud analyst reviews LLM flagged cases → labels → retraining data.


### Design 20: Design an AI-powered data extraction pipeline from unstructured documents.
**A:**
```
Document ingestion (PDF, Word, email, scanned images)
→ Document classification: invoice, contract, form, report
→ Document parsing:
   ├── Digitally-created PDF: pdfplumber + OCR-free text extraction
   └── Scanned/image: Tesseract OCR or Azure Document Intelligence
→ Structure extraction:
   ├── LLM extraction: "Extract: {fields} from this document" → JSON
   ├── Table extraction: Camelot/pdfplumber for structured tables
   └── Key-value extraction: fine-tuned information extraction model
→ Validation: schema validation, business rule checks, confidence scoring
→ Human review queue: low-confidence extractions
→ Output: structured JSON → database / downstream system
```

🏷️ *Nyaya-Pro has a version of this: legal document ingestion → extract section headings, subsection numbers, cross-references → structured index for FAISS retrieval.*


### Design 21: Design a personalized learning assistant.
**A:**
**Core loop:**
```
Learner profile (knowledge level, learning goals, past performance)
→ Adaptive content selection: Bloom's taxonomy → select appropriate difficulty
→ Content delivery: explanation + examples + interactive exercises
→ Assessment: quiz questions (generated by LLM from content)
→ Feedback analysis: identify knowledge gaps from wrong answers
→ Profile update: update learner model (Bayesian Knowledge Tracing or LLM-based)
→ Next content recommendation
```

**Personalisation levers:** difficulty adaptation, learning style (visual/text), pacing, topic sequencing (prerequisite graph), language preference.

**Spaced repetition:** track item memory strength; re-expose items before forgetting (Ebbinghaus forgetting curve).

🏷️ *ExamGenie AI is a primitive version — takes a PDF, generates MCQs. A full personalized learning assistant would add adaptive difficulty, spaced repetition, and learner profiling.*


### Design 22: Design an AI system for automated code migration.
**A:**
```
Source codebase (old Python 2 / Java 8 / etc.)
→ AST parsing: understand code structure (no pure LLM text processing)
→ Dependency analysis: identify external APIs, deprecated functions
→ Migration planning agent: create file-by-file migration plan
→ Per-file migration agents (parallel):
   ├── LLM translates file to target language/version
   ├── AST-based transformations for mechanical changes (more reliable)
   └── Test generation for migrated code
→ Integration testing: run full test suite on migrated codebase
→ Diff review: human reviews changes per file
→ Iteration: fix failing tests (agent loop)
```

**Key insight:** Use AST-based mechanical transforms for predictable changes (API renames, syntax changes). Reserve LLM for semantic translation (complex logic, paradigm shifts). LLM alone on large codebases = expensive and error-prone.


### Design 23: Design an AI-powered legal document review system.
**A:**
```
Legal document ingestion (contract PDF)
→ Document structure extraction: clauses, parties, dates, defined terms
→ Parallel review agents:
   ├── Risk identification agent: flag unusual/risky clauses vs standard
   ├── Completeness checker: are required clauses present?
   ├── Consistency checker: are defined terms used consistently?
   ├── Obligation extractor: what does each party owe?
   └── Comparison agent: diff against template contract
→ Risk scoring: high/medium/low per clause
→ Lawyer review interface: clause-by-clause with AI annotations
→ Negotiation suggestions: "Standard fallback for this clause: [alternative text]"
```

🏷️ *This is the enterprise version of Nyaya-Pro. I'd extend it with: contract comparison (embedding similarity to standard terms), obligation tracking, and integration with Salesforce CRM for contract lifecycle management.*


### Design 24: Design a conversational AI system with memory across sessions.
**A:**
**Memory architecture:**
- Short-term: Redis session store (TTL 30min)
- Long-term: PostgreSQL + vector embeddings (Pinecone)

**Memory update pipeline:**
After each session:
1. Summarise session using LLM
2. Extract key facts: `{"name": "Mangesh", "profession": "AI Engineer", "prefers": "concise answers"}`
3. Store facts in structured DB (easy retrieval by key)
4. Store session summary as embedding in vector DB (semantic retrieval)

**Retrieval at conversation start:**
1. Load user profile (name, preferences, key facts) from structured DB
2. Embed current query → retrieve relevant past conversations from vector DB
3. Build context: user profile + relevant history + current query

**Privacy controls:** users can view, edit, or delete all stored memories.


### Design 25: How do you design for latency vs quality trade-offs in AI systems?
**A:**
**Framework for trade-off decisions:**

| Latency Budget | Recommended Approach |
|----------------|---------------------|
| < 200ms | Cached response, rule-based, or tiny embedded model |
| 200-500ms | Small/fast LLM (Gemini Flash, GPT-4o-mini, Groq) |
| 500ms-2s | Full LLM (GPT-4o, Claude Sonnet), standard RAG |
| 2-10s | Complex RAG with reranking, agentic pipeline |
| 10s+ | Deep research agent, multi-step reasoning |

**Techniques to push quality without sacrificing latency:**
1. Speculative decoding: faster output, same quality
2. Streaming: start displaying output at 200ms even if generation takes 2s
3. Cascade/routing: classify query complexity → route simple queries to fast model, complex to slow
4. Parallel calls: run multiple agents simultaneously rather than sequentially
5. Caching: semantic cache for repeated queries


### Design 26: How do you implement caching strategies for LLM applications?
**A:**
**Three layers of caching:**

1. **Exact response cache:** Hash(prompt) → cached response. Works for identical queries. Redis with TTL.

2. **Semantic cache:** Embed query → find semantically similar past queries (cosine > 0.95) → return their cached response. GPTCache, LangChain SemanticCache.

3. **Prompt prefix cache:** Cache the KV cache of the system prompt + common context. Gemini prefix caching, Anthropic prompt caching. Save ~70% tokens for long system prompts.

**Cache invalidation:**
- TTL-based: expire after N hours/days for dynamic content
- Event-based: invalidate when underlying data changes
- Version-based: cache key includes model version and prompt version

**When not to cache:** personalised responses, real-time data queries, safety-critical responses that must always be freshly generated.

🏷️ *Nyaya-Pro: Gemini prompt caching for 32K-token legal context → 75% token cost reduction. Redis semantic cache for common queries (bail provisions, marriage law) → ~40% of queries served from cache.*


### Design 27: How do you design rate limiting and cost management for AI APIs?
**A:**
**Rate limiting (per user/tenant):**
```
Request → API Gateway → Rate Limiter (Redis + sliding window):
   ├── Token-per-minute limit per user
   ├── Request-per-minute limit
   └── Monthly cost limit per tenant
→ If limit exceeded: 429 Too Many Requests + retry-after header
```

**Cost management:**
1. **Token estimation** — estimate tokens before call; reject if estimated cost > budget
2. **Cost metering** — track every call: user_id, tokens_in, tokens_out, model, cost
3. **Usage dashboards** — per-user, per-feature cost visibility
4. **Budget alerts** — Slack alert when 80% of monthly budget consumed
5. **Automatic throttling** — downgrade to cheaper model when approaching limits
6. **Cost allocation** — tag all LLM calls with feature/team for cost attribution

**Architecture:** API gateway (Kong/custom) handles rate limiting. Usage events → Kafka → cost calculation service → billing DB.


### Design 28: How do you handle failover and fallback strategies for AI systems?
**A:**
**Failover hierarchy:**
```
Request → Primary LLM (GPT-4o)
   → Timeout/error → Secondary LLM (Claude Sonnet) 
   → Timeout/error → Tertiary LLM (Gemini Pro)
   → Timeout/error → Cached response (similar past query)
   → No cache → Rule-based fallback response
   → Last resort → "Service temporarily unavailable" + human handoff
```

**Implementation:**
- Circuit breaker pattern: if primary fails 5× in 60s, open circuit → stop calling for 60s → half-open → test with 1 request
- Retry with exponential backoff: 1s, 2s, 4s, 8s (max 3 retries)
- Health checks: ping each LLM provider every 30s; pre-warm secondary before primary fails
- Fallback response library: pre-written responses for common queries served when all LLMs are down

🏷️ *JobPilot AI: primary = Groq (fast), fallback = Ollama local Llama (always available). If Groq rate-limits, Ollama handles the overflow with slightly degraded latency.*


### Design 29: How do you design an AI system for high availability and fault tolerance?
**A:**
**HA principles applied to AI systems:**

1. **No single point of failure:**
   - Multiple LLM providers (not 100% dependent on one)
   - Multiple inference servers with load balancer
   - Vector DB with read replicas

2. **Graceful degradation:**
   - If LLM is down → return cached/rule-based responses
   - If vector DB is down → answer from LLM knowledge only (with disclaimer)
   - If embedding service is down → disable semantic search, fall back to keyword search

3. **Health checks + auto-recovery:**
   - Liveness probe: is the service running?
   - Readiness probe: can it handle requests? (LLM responding correctly?)
   - Kubernetes auto-restart on failure

4. **Data durability:**
   - Vector DB: cross-AZ replication
   - Session state: Redis with persistence + replica

5. **Chaos engineering:**
   - Simulate LLM provider outage in staging → verify fallbacks work


### Design 30: How do you design an AI system that gracefully degrades when the model is unavailable?
**A:**
**Degradation tiers:**

| Model Status | Degraded Behaviour |
|-------------|-------------------|
| Primary down | → Switch to secondary model automatically |
| All LLMs down | → Serve cached responses for known queries |
| No cache match | → Rule-based responses for common intents |
| No rule match | → "Our AI assistant is temporarily unavailable. We've logged your query and will respond shortly." |
| Complete outage | → Static FAQ page, human support channel |

**Implementation:**
- Cache warm-up: pre-cache responses for the top-100 most common queries
- Deterministic fallbacks: write simple if/else handlers for top user intents
- User communication: clear, honest messaging about degraded mode
- SLO definition: what % of queries must be answered? By which method?
- Monitoring: alert when degradation tier > 0 for more than 5 minutes


### Design 31: Key considerations for multi-region deployment of AI systems.
**A:**
**Data residency:** Some regions (EU, India) require data to stay within borders. You may need separate LLM endpoints and vector DBs per region.

**Model availability:** Not all LLM providers are available in all regions (e.g., OpenAI has region restrictions). Plan your provider mix by region.

**Latency:** Route users to nearest region. Use CDN for static assets, regional API gateways for LLM calls.

**Vector DB replication:** Replicate indices across regions but manage write consistency (eventual consistency is acceptable for read-heavy RAG).

**Prompt caching:** Region-scoped caches (don't route EU prompts to US cache).

**Compliance:** GDPR (EU), PIPL (China), PDPA (Singapore) — different data handling requirements per region.

**Model versioning:** Rolling upgrades per region — test in one region before global rollout.


### Design 32: Design an AI-powered search engine for an e-commerce platform.
**A:**
```
User Query
→ Query understanding: intent (navigational / informational / transactional), entity extraction (brand, category, price range)
→ Query expansion: LLM adds synonyms, related terms
→ Multi-stage retrieval:
   ├── Keyword search (Elasticsearch): exact product name, SKU matches
   ├── Semantic search (FAISS): embedding similarity to product descriptions
   └── Personalised retrieval: boost products from user's category preferences
→ LLM re-ranking: "Rank these 50 products for a user asking '{query}' based on relevance, popularity, and personalisation"
→ Business rules: boost sponsored, filter out-of-stock, boost margin products
→ Results with LLM-generated snippet: "Why this product matches your search"
→ A/B test: measure click-through rate and conversion for different ranking strategies
```


### Design 33: Design an AI gateway/proxy for managing LLM access across an organisation.
**A:**
```
Internal Services → AI Gateway (reverse proxy)
→ Authentication & authorisation (which team can use which model?)
→ Rate limiting (per team/application)
→ Cost tracking & allocation (tag by team, feature, environment)
→ Request/response logging (with PII masking)
→ Caching (semantic + exact match)
→ Model routing:
   ├── Route by capability: code → Claude, chat → GPT-4o, fast → Groq
   └── Route by cost: dev env → cheaper models, prod → quality models
→ Fallback management: primary → secondary on failure
→ Output filtering: remove PII, harmful content from responses
→ Admin dashboard: usage, cost, errors per team
```

Benefits: centralised governance, cost visibility, consistent observability, ability to swap providers without changing application code.


### Design 34: How do you design a RAG system that handles conflicting information across sources?
**A:**
1. **Surface the conflict explicitly:** Instead of silently picking one answer, tell the user: "Note: Source A (2024) says X, while Source B (2022) says Y."
2. **Source authority ranking:** Assign credibility scores (official legislation > academic > news > blog). Prefer high-authority sources.
3. **Recency weighting:** More recent sources preferred for evolving topics (legislation, medical guidelines).
4. **Consensus detection:** If 3 of 4 sources agree, present the majority view with the dissenter noted.
5. **Provenance transparency:** Always show which source supports which claim.
6. **Conflict resolution prompt:** "These sources contain conflicting information. Which is more authoritative and why? Synthesise the most accurate answer."
7. **Human escalation:** For high-stakes domains (medical, legal), flag conflicts for expert review rather than auto-resolving.


### Design 35: How do you approach capacity planning for an AI system?
**A:**
**What to estimate:**
1. **QPS (Queries Per Second)**: historical data or projections
2. **Token estimate**: avg input tokens per query × QPS = tokens/second demand
3. **Latency budget**: what's your P95 latency target?
4. **Model throughput**: benchmark your model/infra stack for tokens/second per GPU

**Capacity formula:**
```
GPUs needed = (tokens/sec demand × avg_latency_s) / (tokens/GPU/s × target_GPU_utilisation)
```

**Buffer:** always provision 2× peak demand capacity (traffic spikes).

**Auto-scaling:** Kubernetes HPA on GPU utilisation and queue depth metrics.

**Cost projection:**
```
Monthly cost = GPUs × GPU_price/hr × 720 hours + (API tokens × per_token_price)
```

**Cost optimisation:** Spot instances (70% cheaper, handle preemption), right-size GPU type (A100 for generation, T4 for embedding).


### Design 36: Design a multi-tenant AI chatbot platform where each business gets a custom chatbot.
**A:**
**Architecture:**
```
Tenant onboarding:
→ Upload knowledge base (PDFs, URLs, text)
→ Configure: persona, tone, restricted topics, escalation rules
→ Index documents → isolated namespace in vector DB

Request handling:
→ Request arrives with tenant_id
→ Load tenant config (system prompt, persona, escalation rules)
→ Retrieve from tenant's namespace only
→ Generate with tenant's custom persona
→ Apply tenant-specific guardrails
→ Response
```

**Tenant isolation:** separate vector DB namespaces, separate config stores, separate conversation histories in PostgreSQL.

**Customisation options:** custom system prompt, logo and brand colours in UI, escalation email/webhook, restricted topic list, custom FAQ entries.

**Billing:** per conversation, per token, or per month subscription. Track per-tenant token consumption.

**Admin dashboard:** each tenant sees their own usage, conversation logs, and can update their knowledge base.


### Design 37-42: Additional Design Problems

### Design 37: AI Meeting Summariser for Thousands of Meetings Daily
**A:** Audio → Whisper STT → speaker diarisation → structured transcript → LLM summarisation (key decisions, action items, attendees) → email + Slack delivery. Async pipeline; Kafka for ingestion; GPU cluster for transcription; LLM for summarisation; Redis for deduplication.

---

### Design 38: AI Notification System that Prioritises Instead of Broadcasting
**A:** User notification history + current notifications → ML ranker (user engagement features) → LLM: "Given this user's context, which of these notifications are relevant right now?" → priority-sorted delivery → A/B test notification strategies by measuring engagement.

---

### Design 39: AI Anomaly Detection for Cloud Infrastructure
**A:** Metrics stream (Prometheus) → time-series model (Isolation Forest, LSTM) for baseline → LLM for alert interpretation: "CPU spike + memory spike + increased error rate = likely memory leak in service X" → Pager duty alert with LLM-generated runbook.

---

### Design 40: AI Document Processing Pipeline for Financial Institutions
**A:** Document ingestion → OCR (Azure DI) → classification (invoice/statement/contract) → extraction (amounts, dates, entities) → validation → human review for low-confidence → database → compliance audit trail. HIPAA/SOC2 compliant logging.

---

### Design 41: AI Dynamic Pricing Engine
**A:** Real-time demand signals + competitor prices + inventory levels → LLM reasoning: "Given these market conditions, justify pricing adjustment" → price update with explanation → human approval for >10% changes → A/B test pricing strategies.

---

### Design 42: AI Resume Screening (100K applications/week)
**A:** Bulk ingestion pipeline → resume parsing (LLM structured extraction) → embedding for semantic JD matching → LLM scoring (skills match, experience relevance) → bias audit (remove name/gender/age before scoring) → ranked shortlist → human reviewer sees ranked list with LLM reasoning.

🏷️ *JobPilot AI is the candidate-side of this — it optimises the resume to score better on exactly such systems.*
